**`validate_nc_permit_occupancy_evidence`**

Ingests North Carolina building-permit records (Shovels), links them to footprints via `US_footprint-spine-2026`, and inspects the resulting `occupancy_type_property_shovels` evidence column as an independent cross-check for occupancy_type imputation testing.

## Source data

Two Shovels-exported parquet files sit under `data/external/US/NC/_all/building/shovels/permits/`: `Permit.parquet` (91 columns, 4,435,881 rows -- one row per permit) and `Permit_C.parquet` (a byte-identical 77-column subset, not used). `Permit.parquet` was copied to this recipe's expected external path, `data/external/US/NC/_all/property/shovels/2026/Permit.parquet`, and is ingested by `US-NC_property-shovels-2026` (`entity_type: property`, since a permit is a work-event snapshot on a property -- the same shape as Florida's annual DOR tax-roll snapshots -- not an ownership transfer like `entity_type: transaction`).

Linkage to footprints happens in two passes on `US_footprint-spine-2026`, APN first: `link_by_id` on `parcel_id_local` (both sides compute this standardized key at ingest time), then an address-based fallback via `address_id_local` (`derive_address_id_local`, joined through `US_property-spine-2026` so the permit source's address gets the same town-scoped key as every other property source). The evidence column, `occupancy_type_property_shovels`, is mapped from Shovels' own `PROPERTY_TYPE_DETAIL` classification via a sidecar remap CSV onto the same `occupancy_type` category set used everywhere else in openplaces.

`occupancy_type_property_shovels` is **not** yet wired into `US_footprint-cheer-2026`'s occupancy vote -- that vote/kernel mechanism is mid-overhaul on a separate worktree. This notebook validates the evidence column on its own merits first.

# Configure

In [ ]:
import argparse

import pandas as pd

import openplaces as op

In [ ]:
parser = argparse.ArgumentParser(
    description='Validate NC permit occupancy_type evidence'
)
parser.add_argument('--recipe_id', default='US-NC_property-shovels-2026')
parser.add_argument('--footprint_recipe_id', default='US_footprint-spine-2026')
parser.add_argument('--admin_ids', nargs='*')
parser.add_argument('--reprocess', action='store_true')
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = (
    '--recipe_id US-NC_property-shovels-2026 '
    '--footprint_recipe_id US_footprint-spine-2026 '
    '--admin_ids US-NC-NE '
    '--reprocess '
    '--verbose '
)
args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

# Ingest and harmonize

In [ ]:
op.ingest(
    args.recipe_id,
    'US-NC',  # parquet source has no FID chunking: always ingest whole-state
    reprocess=args.reprocess,
    verbose=args.verbose,
)

In [ ]:
# The address-fallback pass links through US_property-spine-2026, which
# must be re-harmonized first so it picks up this source's rows.
op.harmonize(
    'US_property-spine-2026',
    args.admin_ids,
    reprocess=args.reprocess,
    verbose=args.verbose,
)
op.harmonize(
    args.footprint_recipe_id,
    args.admin_ids,
    reprocess=args.reprocess,
    verbose=args.verbose,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True

convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=COMMIT)

# Inspect outputs

In [ ]:
footprints = op.get_entities(args.footprint_recipe_id, args.admin_ids)
print(len(footprints))
footprints.sample(5).T

## occupancy_type_property_shovels coverage

APN-linked rows (`n_permits_per_footprint > 0`) should dominate; address-fallback rows (`n_permits_per_footprint_address > 0` but no APN match) fill in the remainder.

In [ ]:
col = 'occupancy_type_property_shovels'
print(f'{col} fill rate: {footprints[col].notna().mean():.1%}')
footprints[col].value_counts(dropna=False)

In [ ]:
apn_matched = footprints['n_permits_per_footprint'] > 0
address_matched = footprints['n_permits_per_footprint_address'] > 0
print(f'APN-matched: {apn_matched.mean():.1%}')
print(f'Address-matched: {address_matched.mean():.1%}')
print(
    'Address-only (APN missed, address filled): '
    f'{(address_matched & ~apn_matched).mean():.1%}'
)

## Agreement with existing occupancy evidence

Cross-tab against the harmonized spine's other occupancy-related evidence columns -- not a formal accuracy check (no ground truth here), just a sanity check that the new source's Single-Family / Multi-Family / non-residential split lines up with what NSI and the parcel land-use group already suggest.

In [ ]:
for other_col in [
    'occupancy_type_building_nsi',
    'group_parcel',
]:
    if other_col not in footprints.columns:
        continue
    both = footprints[[col, other_col]].dropna()
    print(f'\n{other_col}: {len(both)} footprints with both evidence sources')
    print(pd.crosstab(both[col], both[other_col]))

## Raw permit table

In [ ]:
permits = op.get_entities(args.recipe_id, args.admin_ids[0])
print(len(permits))
print(f'parcel_id_local fill: {permits["parcel_id_local"].notna().mean():.1%}')
print(f'occupancy_type_raw fill: {permits["occupancy_type_raw"].notna().mean():.1%}')
permits.sample(5).T